<a href="https://colab.research.google.com/github/bradleyfriesen/job_shadow_jimi/blob/main/job_shadow_2026_with_jimi_brad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Welcome to the Prep Notebook! If you are:**
* new to Python? watch: https://www.youtube.com/watch?v=kqtD5dpn9C8
* new to Google Collab? watch: https://www.youtube.com/watch?v=RLYoEyIHL6A
* new to Pandas libray inside Python? watch: https://youtu.be/2uvysYbKdjM?si=3uOBxU9CGqx63KjG
* new to other Python data science related libraries like seaborn, sklearn...google/youtube/chatgpt them individually to get a rough sense of what each one can do!

Once you are equiped with basic understanding of the platform/language/tools mentioned above, **make a copy of this collab notebook** and add the copy to your own google drive so you can work off there and keep it forever :)

NEXT, let's start by undersanding the business ask and the dataset we are given!


---


##### *The Ask: Working as the Data Scientist for Toronto Parking Authority (City Government), you are asked to give a 5-minute, 3-slide presentation to Toronto Mayor Olivia Chow on what is the most important ONE recommendation Toronto Bike Share needs to do next, based on 2025 data, in order to aid the city's TransformTO climate goals, and provide reasoning.*


---


**TransformTO: https://www.toronto.ca/services-payments/water-environment/environmentally-friendly-city-initiatives/transformto/**

**Dataset: https://open.toronto.ca/dataset/bike-share-toronto-ridership-data/**

# Part 1: Environment Setup & Data Extraction (Completed)

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import requests
import zipfile
import io
import glob
import os
import time

# Get metadata for the Bike Share Toronto dataset
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/package_show"
params = { "id": "bike-share-toronto-ridership-data"}
package = requests.get(base_url, params=params).json()

# Find the resource for 2025 (looking for the ZIP file)
resources = package["result"]["resources"]
target_resource = next(r for r in resources if "2025" in r["name"] and r["format"] == "ZIP")

# Download and extract
print(f"Downloading: {target_resource['name']}")
r = requests.get(target_resource["url"])
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("./bikeshare_2025")

# Load all CSVs from the extracted folder into one DataFrame
all_files = glob.glob("./bikeshare_2025/*.csv")
df_trips = pd.concat(
    (pd.read_csv(f, encoding='latin1') for f in all_files),
    ignore_index=True
)

print(f"Loaded {len(df_trips):,} rows from 2025 data.")
df_trips.head()

Downloading: bikeshare-ridership-2025
Loaded 7,083,646 rows from 2025 data.


,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,34873865,455,7199.0,2025-02-01 00:00:49,College St / Markham St,7206.0,2025-02-01 00:08:24,College St / Markham St,1927,Member,ICONIC
1,34873866,755,7032.0,2025-02-01 00:01:31,Augusta Ave / Dundas St W,7675.0,2025-02-01 00:14:06,Augusta Ave / Dundas St W,3083,Member,ICONIC
2,34873874,674,7248.0,2025-02-01 00:05:51,Baldwin St / Spadina Ave - SMART,7297.0,2025-02-01 00:17:05,Baldwin St / Spadina Ave - SMART,6307,Member,ICONIC
3,34873869,197,7239.0,2025-02-01 00:01:57,Bloor St W / Manning Ave - SMART,7192.0,2025-02-01 00:05:14,Bloor St W / Manning Ave - SMART,1433,Member,ICONIC
4,34873879,629,7929.0,2025-02-01 00:07:26,Spadina Ave / Bulwer St- SMART,7524.0,2025-02-01 00:17:55,Spadina Ave / Bulwer St- SMART,3588,Member,ICONIC


In [ ]:
df_trips.Start_Time.min(),df_trips.Start_Time.max()

('2025-01-01 00:01:51', '2025-10-31 23:59:59')

In [ ]:
# --- DATASET: Station Information ---
print("\nFetching Station Information...")

# Fetch real-time station coordinates from the global GBFS feed
station_info_url = "https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
response_stations = requests.get(station_info_url).json()
df_stations = pd.DataFrame(response_stations['data']['stations'])

# Keep only necessary spatial columns (ID, latitude, longitude, name)
df_stations = df_stations[['station_id', 'lat', 'lon', 'name']]
print("✅ Station Information loaded successfully! (Stored as: df_stations)")
df_stations.head()


Fetching Station Information...
✅ Station Information loaded successfully! (Stored as: df_stations)


,station_id,lat,lon,name
0,7000,43.639832,-79.395954,Fort York Blvd / Capreol Ct
1,7001,43.664964,-79.383550,Wellesley Station Green P
2,7002,43.667131,-79.399555,St. George St / Bloor St W
3,7003,43.667018,-79.402796,Madison Ave / Bloor St W
4,7005,43.648001,-79.383177,King St W / York St


In [ ]:
df_stations.station_id.nunique(),len(df_stations)

(1008, 1008)

In [ ]:
# --- DATASET: Historical Weather Data (All of 2025) ---
print("\nFetching 2025 Hourly Weather Data from Environment Canada...")
weather_frames = []

# Loop through all 12 months to pull historical weather
for month in range(1, 13):
    weather_url = (
        "https://climate.weather.gc.ca/climate_data/bulk_data_e.html?"
        f"format=csv&stationID=51459&Year=2025&Month={month}&Day=1&timeframe=1&submit=Download+Data"
    )
    df_weather_month = pd.read_csv(weather_url)
    weather_frames.append(df_weather_month)
    # A brief pause to avoid overloading the government server
    time.sleep(0.5)

# Stitch all weather months together
df_weather = pd.concat(weather_frames, ignore_index=True)
print(f"✅ 2025 Historical Weather loaded! (Stored as: df_weather with {len(df_weather):,} rows)")

print("\n🎉 All data is ready! You may now begin the Data Cleaning & Feature Engineering phase.")
df_weather.head()


Fetching 2025 Hourly Weather Data from Environment Canada...
✅ 2025 Historical Weather loaded! (Stored as: df_weather with 8,760 rows)

🎉 All data is ready! You may now begin the Data Cleaning & Feature Engineering phase.


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time (LST),Year,Month,Day,Time (LST),Flag,Temp (°C),Temp Flag,Dew Point Temp (°C),Dew Point Temp Flag,Rel Hum (%),Rel Hum Flag,Precip. Amount (mm),Precip. Amount Flag,Wind Dir (10s deg),Wind Dir Flag,Wind Spd (km/h),Wind Spd Flag,Visibility (km),Visibility Flag,Stn Press (kPa),Stn Press Flag,Hmdx,Hmdx Flag,Wind Chill,Wind Chill Flag,Weather
0,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01 00:00,2025,1,1,00:00,NaN,1.6,NaN,1.6,NaN,100.0,NaN,NaN,NaN,35.0,NaN,27.0,NaN,16.1,NaN,97.93,NaN,NaN,NaN,NaN,NaN,Rain
1,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01 01:00,2025,1,1,01:00,NaN,1.6,NaN,1.6,NaN,100.0,NaN,NaN,NaN,35.0,NaN,29.0,NaN,19.3,NaN,97.95,NaN,NaN,NaN,NaN,NaN,Cloudy
2,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01 02:00,2025,1,1,02:00,NaN,1.4,NaN,1.4,NaN,100.0,NaN,NaN,NaN,35.0,NaN,30.0,NaN,19.3,NaN,97.98,NaN,NaN,NaN,NaN,NaN,"Rain,Snow"
3,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01 03:00,2025,1,1,03:00,NaN,0.8,NaN,0.8,NaN,100.0,NaN,NaN,NaN,34.0,NaN,26.0,NaN,9.7,NaN,98.03,NaN,NaN,NaN,NaN,NaN,"Rain,Snow,Fog"
4,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01 04:00,2025,1,1,04:00,NaN,1.4,NaN,1.4,NaN,100.0,NaN,NaN,NaN,34.0,NaN,23.0,NaN,19.3,NaN,98.07,NaN,NaN,NaN,NaN,NaN,Rain


In [ ]:
df_weather['Date/Time (LST)'].min(),df_weather['Date/Time (LST)'].max()

('2025-01-01 00:00', '2025-12-31 23:00')

**What type of answer is my boss expecting? What does some of these columns mean? Welcome to a taster of how ambigious and messy real-life can be sometimes!**

If you want to start exploring sooner, go right ahead :)

# Part 2: Combine, Denoise, and Gather Actionable Insights


---

### There is a infinite number of things we could explore next with these 3 datasets, remember, there are no right directions/questions/answers here, so, explore away!

# Part Bonus: Above and Beyond


---


### Should we proactively dispatch rebalancing trucks to high-demand stations before the 5:00 PM rush hour?